# Compute Routing

## What you'll learn

- Understand the difference between dispatch backends and compute routing
- Switch a step's compute target with the `compute` parameter
- Set pipeline-wide defaults with `default_compute`
- Mix compute targets in a single pipeline
- Validate operations for remote execution

**Prerequisites:** [First Pipeline](../getting-started/01-first-pipeline.ipynb),
[Step Overrides](06-step-overrides.ipynb).
**Estimated time:** 15 minutes
**GPU required:** No.

In [1]:
from __future__ import annotations

from artisan.execution.compute import validate_remote_execute
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [2]:
env = tutorial_setup("compute_routing")

## Dispatch vs compute routing

Artisan has two orthogonal axes for controlling where work runs:

```
  Dispatch backend              Compute target
  (where the worker runs)       (where execute() runs inside the worker)

  ┌─────────────────┐           ┌─────────────────┐
  │  Local process   │──────────│  Local call      │  (default)
  │  SLURM job       │──────────│  Modal container │
  │  SLURM intra     │──────────│  (future)        │
  └─────────────────┘           └─────────────────┘
```

| Axis | Controls | Parameter | Example |
|------|----------|-----------|--------|
| Dispatch backend | Where the *worker process* runs | `backend` | `Backend.LOCAL`, `Backend.SLURM` |
| Compute target | Where *execute()* runs inside the worker | `compute` | `"local"`, `"modal"` |

These axes are independent. A SLURM worker can route execute() to Modal.
A local worker can also route execute() to Modal. The dispatch backend
handles scheduling and sandbox setup; compute routing handles only the
execute() call itself.

## The `compute` parameter

Every `pipeline.run()` and `pipeline.submit()` call accepts a `compute`
parameter. Set it to route execute() to a different target. Everything
else — operation class, inputs, params, output wiring — stays identical.

```python
# Local (default)
step = pipeline.run(MyOp, inputs=..., compute="local")

# Modal — same operation, same inputs, different compute target
step = pipeline.run(MyOp, inputs=..., compute="modal")
```

This means you can develop and debug locally, then move heavy compute to
Modal by changing one argument per step.

In [ ]:
pipeline = PipelineManager.create(
    name="compute_routing_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_compute="local",
)
output = pipeline.output

print(f"Pipeline created with default_compute='{pipeline.config.default_compute}'")

### Generate data (local compute)

DataGenerator is fast — runs locally with `compute="local"`.

In [ ]:
step0 = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 4, "seed": 42},
    compute="local",
)
print(f"Generated {step0.succeeded_count} datasets with local compute")

### Transform data

In production, you would use `compute="modal"` for a GPU-intensive step.
Here we use `"local"` to demonstrate the API without requiring credentials.
The `compute` parameter follows the same override pattern as `backend`.

In [ ]:
step1 = pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 0.5, "variants": 2, "seed": 100},
    compute="local",
)
print(f"Transformed {step1.succeeded_count} datasets with local compute")

In [ ]:
step2 = pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
    compute="local",
)
print(f"Computed metrics for {step2.succeeded_count} datasets with local compute")

In [ ]:
summary = pipeline.finalize()
print(
    f"Pipeline complete: {summary['total_steps']} steps, "
    f"success={summary['overall_success']}"
)
inspect_pipeline(env.delta_root)

## Pipeline-wide defaults with `default_compute`

Set `default_compute` at pipeline creation to apply a compute target to
all steps by default. Individual steps can still override with the
`compute` parameter.

Override precedence follows the same pattern as `backend`:

```
Pipeline default  →  Operation class default  →  Step override (wins)
```

In [ ]:
pipeline2 = PipelineManager.create(
    name="compute_defaults_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_compute="local",
)
output2 = pipeline2.output

# These steps inherit default_compute="local" — no explicit compute needed
pipeline2.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 3, "seed": 42},
)
pipeline2.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output2("generate", "datasets")},
)

# This step explicitly overrides the default
pipeline2.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output2("transform", "dataset")},
    compute="local",  # explicit override (would be "modal" in production)
)

pipeline2.finalize()
print(f"Pipeline default_compute: '{pipeline2.config.default_compute}'")
print("All steps completed — first two inherited the default, last one overrode it")

## Validating operations for remote execution

`validate_remote_execute()` is a pre-flight check for operations that
will run on remote compute targets. It verifies:

- **Cloudpickle round-trip** — serializes and deserializes the operation
  instance. Fails if the operation has unpicklable attributes (file
  handles, lambdas, etc.).
- **ToolSpec path check** — warns if `tool.executable` points to a
  local-only path that won't exist on the remote container.

Use it in test suites and before deploying to remote providers:

```python
assert validate_remote_execute(MyOp())
```

In [ ]:
gen = DataGenerator()
trans = DataTransformer()

print(f"DataGenerator:   validate_remote_execute = {validate_remote_execute(gen)}")
print(f"DataTransformer: validate_remote_execute = {validate_remote_execute(trans)}")
print("\nBoth operations are safe for remote compute routing.")

## Summary

| Concept | What it does |
|---------|-------------|
| Dispatch backend (`backend`) | Controls where the *worker process* runs (local, SLURM) |
| Compute target (`compute`) | Controls where *execute()* runs inside the worker (local, Modal) |
| `default_compute` | Pipeline-wide default compute target |
| `validate_remote_execute()` | Pre-flight check for cloudpickle and tool path compatibility |

Dispatch and compute routing are orthogonal — any combination works.
Develop and debug with `compute="local"`, then switch to `"modal"` for
GPU work by changing one argument per step.

## Next steps

- [Running on Modal](14-modal-execution.ipynb) — Configure Modal containers, GPU selection, and sandbox transport
- [Step Overrides](06-step-overrides.ipynb) — All `pipeline.run()` override parameters
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work
- [Configure Execution](../../how-to-guides/configuring-execution.md) — Complete configuration reference